# TRAIN THE MODEL

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt  #visualization
from tensorflow.keras.preprocessing.image import ImageDataGenerator #Prevents overfitting
from tensorflow.keras.applications import MobileNetV2 # used for image classification, light weight.  
from tensorflow.keras.models import Sequential, load_model     #loads previous model 
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D #a fully connected neural network #reduce the height and width 
from tensorflow.keras.optimizers import Adam #adam is a optimization algorithm that is used to update the weights of the neural network
import numpy as np
from tensorflow.keras.preprocessing import image#image processing

# Function to extract frames from each video
def extract_frames(video_path, output_folder, fps=10): #extracts frames from the video
    cap = cv2.VideoCapture(video_path)                 #captures the video from the path given
    video_fps = cap.get(cv2.CAP_PROP_FPS)              #gets the frames per second of the video
    frame_interval = int(video_fps / fps)           #calculates the frame interval      
    frame_count = 0                                         #initializes the frame count to 0
    
    while True:
        ret, frame = cap.read() #ret is a boolean value whether the frame was successfully read or not , frame is the image
        if not ret:
            break
        if frame_count % frame_interval == 0:
            frame_filename = os.path.join(output_folder, f"frame_{frame_count}.jpg") #joins the path of the output folder and the frame number
            cv2.imwrite(frame_filename, frame)  #saves the frame in the output folder with
        frame_count += 1    #increases the frame count by 1
    cap.release()           #releases the video capture object

# Function to extract frames from all videos in a folder
def extract_frames_from_folder(video_folder, output_folder, fps=10): #extracts frames from all the videos in the folder
    for video_filename in os.listdir(video_folder):
        if video_filename.endswith('.mp4'): #checks if the file is a video file
            video_path = os.path.join(video_folder, video_filename)   #joins the path of the video folder and the video filename
            print(f"Processing video: {video_path}")                  #prints the video path
            video_name = os.path.splitext(video_filename)[0]          #splits the video filename into the name and the extension
            video_output_folder = os.path.join(output_folder, video_name)       
            os.makedirs(video_output_folder, exist_ok=True)       #creates the output folder
            extract_frames(video_path, video_output_folder, fps)  #extracts the frames from the video

# Paths to the suspicious and normal video folders
suspicious_folder = r"C:\Users\rohan\Desktop\videos and code\Shoplifting dataset\suspicious"
normal_folder = r"C:\Users\rohan\Desktop\videos and code\Shoplifting dataset\normal"
suspicious_output_folder = r"C:\Users\rohan\Desktop\videos and code\Shoplifting dataset\output\suspicious"
normal_output_folder = r"C:\Users\rohan\Desktop\videos and code\Shoplifting dataset\output\normal"

# Extract frames
extract_frames_from_folder(suspicious_folder, suspicious_output_folder, fps=10)
extract_frames_from_folder(normal_folder, normal_output_folder, fps=10)

print("Frame extraction completed.")

# Prepare Data
train_datagen = ImageDataGenerator(         #to prevent overfitting
    rescale=1./255,                         #rescale the image
    validation_split=0.2,                      #split the data into training and validation 80% training 20% validation
    rotation_range=10,                       #rotate the image
    horizontal_flip=True,                 #flip the image               
)

train_generator = train_datagen.flow_from_directory( #flow from directory is used to load the images from the directory
    r"C:\Users\rohan\Desktop\videos and code\Shoplifting dataset\output",  #parht of the directory where the images are to be stored
    target_size=(128, 128),     #resize the image to 128x128
    batch_size=64,              #batch size is the number of images that are passed through the network at once
    class_mode='binary',        #binary classification(is it a normal or suspicious shopper)       
    subset='training'           #whter the data is training or validation(training data)
)

validation_generator = train_datagen.flow_from_directory(#flow from directory is used to load the images from the directory
    r"C:\Users\rohan\Desktop\videos and code\Shoplifting dataset\output",   #path of the directory where the images are to be stored
    target_size=(128, 128),
    batch_size=64,                   
    class_mode='binary',
    subset='validation'             #whether the data is training or validation(validation data)
)

print("Data prepared for training.")

# Build Model
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(128, 128, 3))  #input shape is the shape of the image that is passed through the network
#MobileNetV2 is a pre-trained model that is used for image classification   #include_top=False means that the top layer of the model is not included
base_model.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(1024, activation='relu'),     
    Dense(1, activation='sigmoid')      #sigmoid activation function is used for binary classification 
])  #dense layer is a fully connected neural network

model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])#
#                                # 0 or 1 compare krta hai      #accuracy measure krta hai 
# Train the model
history = model.fit(
    train_generator,
    epochs=5, #epocs are the number of times the model is trained on the data
    validation_data=validation_generator
)

# Save the trained model
model.save('suspicious_shopper_model.h5')  #saves the model
print("Model saved as 'suspicious_shopper_model.h5'")

print("Model training completed.")

# Load the saved model
loaded_model = load_model('suspicious_shopper_model.h5')
print("Model loaded successfully!")

# Evaluate the model
loss, accuracy = loaded_model.evaluate(validation_generator)
print(f'Validation Loss: {loss}')
print(f'Validation Accuracy: {accuracy}')


# Plot Training History (Loss and Accuracy)
def plot_training_history(history):
    # Loss graph
    plt.figure(figsize=(12, 5))                                             #size of the graph
    plt.subplot(1, 2, 1)                                                                                   
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    # Accuracy graph
    plt.subplot(1, 2, 2)
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.show()

plot_training_history(history)    #plots the training history





#  ------------------LIBRARIES ------------------

In [ ]:
import os
import cv2
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt



#  ------------------LOADING THE MODEL ------------------

In [ ]:

# Load the pre-trained model
model_path = "suspicious_shopper_model.h5"
model = load_model(model_path)
print("Model loaded successfully!")

# Function to predict on a single image
def predict_single_image(img_path, model, target_size=(128, 128)):
    img = image.load_img(img_path, target_size=target_size)
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension
    img_array /= 255.0  # Normalize

    prediction = model.predict(img_array)
    if prediction[0] > 0.5:
        return "Suspicious shopper detected", prediction[0][0]
    else:
        return "Normal shopper detected", prediction[0][0]


Model loaded successfully!


#  ------------------Predict on video------------------

In [ ]:


# Function to predict on a video
def predict_from_video(video_path, model, output_size=(128, 128)):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("Error: Could not open video.")
        return

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Preprocess the frame
        frame_resized = cv2.resize(frame, output_size) #resize the frame
        img_array = image.img_to_array(frame_resized) #converts the image to an array
        img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension
        img_array /= 255.0  # Normalize

        # Predict using the model
        prediction = model.predict(img_array, verbose=0)
        prediction_value = prediction[0][0]

        # Determine label and color
        label = 'Suspicious' if prediction_value > 0.5 else 'Normal'#if the prediction value is greater than 0.5 then the label is suspicious otherwise normal
        color = (0, 0, 255) if label == 'Suspicious' else (0, 255, 0)   
                # 0,0,255 is red color and 0,255,0 is green color
        # Display label and confidence on the frame
        cv2.putText(frame, f'{label}: {prediction_value:.2f}', (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2, cv2.LINE_AA)#putting text on the frame
        cv2.imshow("Video Frame", frame)                       #50,50 is the position of the text

        # Quit video with 'q'
        if cv2.waitKey(1) & 0xFF == ord('q'):     #if the key pressed is q then the video will 
            break

    cap.release()        #releases the video capture object
    cv2.destroyAllWindows()  #closes all the windows


## ------------------image and video example usage


In [ ]:


# Example usage
if __name__ == "__main__":
    # Predict on a single image
    image_path = r"C:\Users\rohan\Desktop\REPORT.pdf"
    label, confidence = predict_single_image(image_path, model)
    print(f"Prediction: {label} in image ")

     #Predict on a video
    #video_path = r"C:\Users\rohan\Desktop\Recording 2025-01-01 210346.mp4"
    #print("Press 'q' to quit video playback.")
    #predict_from_video(video_path, model)


UnidentifiedImageError: cannot identify image file <_io.BytesIO object at 0x000002108FF63FB0>

# ------------------------------------FOR LIVE FEEDS------------------------------------

In [ ]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image

# Load the pre-trained model
model_path = "suspicious_shopper_model.h5"
model = load_model(model_path)
print("Model loaded successfully!")

# Function to process live feed
def predict_from_live_feed(model, output_size=(128, 128)):

    cap = cv2.VideoCapture(0)  # 0 for default camera

    if not cap.isOpened():
        print("Error: Could not open webcam.")
        return

    print("Press 'q' to exit the live feed.")

    while True:
        ret, frame = cap.read()
        if not ret:
            print("Error: Unable to read frame.")
            break

        # Resize frame to match model input
        frame_resized = cv2.resize(frame, output_size)

        # Convert the frame to an image format suitable for prediction
        img_array = image.img_to_array(frame_resized)
        img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension
        img_array /= 255.0  # Normalize the image

        # Make prediction on the frame
        prediction = model.predict(img_array, verbose=0)

        # Get the scalar prediction value
        prediction_value = prediction[0][0]

        # Add prediction text on the frame
        label = 'Suspicious' if prediction_value > 0.5 else 'Normal'
        color = (0, 0, 255) if label == 'Suspicious' else (0, 255, 0)
        
        # Display text on the frame
        cv2.putText(frame, f'{label}: {prediction_value:.2f}', (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2, cv2.LINE_AA)

        # Display the frame with prediction
        cv2.imshow("Live Feed", frame)

        # Press 'q' to quit the live feed
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # Release the webcam and close all OpenCV windows
    cap.release()
    cv2.destroyAllWindows()

# Run the live feed prediction
predict_from_live_feed(model)
